# Part e
### Introducing L1 and L2 terms

In [38]:
from pathlib import Path
import sys


here = Path.cwd()
candidates = [here] + list(here.parents)
for p in candidates:
    if (p / "Code").is_dir():
        sys.path.insert(0, str(p))
        break
else:
    raise RuntimeError("Couldn't find a 'Code' folder in this project.")



import autograd.numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from Code.ffnn2 import NeuralNetwork as FFNN 
from Code.scheduler import Adam, RMS_prop, Constant, Scheduler
from Code.data import runge_function, make_data
from Code.cost import CostOLS, dCostOLS
from Code.comparison import run_comparison
from Code.plot import plot_sweep_heatmap, plot_activation_sweep_heatmap
from Code.activations import sigmoid, RELU, LRELU, identity, derivate
from Code.scheduler import Constant, RMS_prop, Adam
from Code.architectures import build_architectures, build_architectures_exact
from Code.helpers import build_nn, make_sched, train_eval_once, sweep, top_k, best_per
%matplotlib inline


import pandas as pd
import seaborn as sns 
sns.set_theme(style="white", font_scale=1.3)
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 200
plt.rcParams.update({
    "savefig.dpi": 200,
    "savefig.bbox": "tight"
})

rho_val, rho2_val = 0.9, 0.999
optimizers_to_sweep = {
    'GD':       (Constant, {}),
    'SGD':      (Constant, {}),      
    'RMS_prop': (RMS_prop, {'rho': rho_val}),
    'Adam':     (Adam,     {'rho': rho_val, 'rho2': rho2_val}),
}

activation_tests = {
    'Sigmoid': sigmoid,
    'RELU': RELU,
    'LRELU': LRELU
}


In [39]:
#Initialize data

(seed_fixed,
 rng_fixed,
 X_fixed_test,
 y_fixed_test,
 N,
 rng,
 x,
 noise,
 y,
 X_train,
 X_val,
 y_train,
 y_val,
 scaler,
 X_train_scaled,
 X_val_scaled,
 X_fixed_test_scaled) = make_data(seed_fixed=42, N=200, noise_std=0.1, test_N=2000)



In [40]:
eta_vals = np.logspace(-4, -1, 4) # Sweep 4 values
lam_vals = [1e-4, 1e-3, 1e-2, 0] #l2
l1_vals = [1e-4, 1e-3, 1e-2, 0 ] #l1
epochs_sweep = 10 #less epochs because of the large number of runs

input_nodes = X_train_scaled.shape[1] 

lam = 0.0
batches = 32


import itertools


In [41]:
depths = [0, 1, 2]          
widths = [50, 100, 200] 

""" architectures_to_sweep = build_architectures(
    depths=(0,1,2), widths=(50,100), out_dim=1,
    two_layer_pairs=[(50,100)]) """


#architectures_to_sweep = build_architectures(depths, widths, out_dim=1)

architectures_to_sweep = build_architectures_exact(out_dim=1)


## L2 regularization

In [42]:

print("sizes:",
      len(architectures_to_sweep),
      len(optimizers_to_sweep),
      len(activation_tests),
      len(eta_vals),
      len(lam_vals))

combos = list(itertools.product(
    architectures_to_sweep.items(),
    activation_tests.items(), 
    optimizers_to_sweep.items(),
    eta_vals, lam_vals
))
print("total combos:", len(combos))

import random
random.seed(42)
sample = random.sample(combos, k=min(120, len(combos)))
print("sample size:", len(sample))


#To get an idea of runtime
import math, time
total = len(architectures_to_sweep) * len(activation_tests) * len(optimizers_to_sweep) * len(eta_vals) * len(lam_vals)
print("Total runs:", total)


L2_sweep_results = {}

print("\n\n================ L2 (Ridge) Regularization with Activation Sweep ================")

for arch_name, layer_sizes in architectures_to_sweep.items():
    print(f"\n---- Architecture: {arch_name} ----")
    act_bucket = {}  # activation -> optimizer -> (len(eta) x len(lam)) grid
    n_hidden_layers = max(0, len(layer_sizes) - 1)

    # === Iterate over activation functions ===
    for act_name, h_func in activation_tests.items():
        print(f"  >> Activation: {act_name}")

        # Hidden activations + identity output (regression)
        activation_funcs = [h_func]*n_hidden_layers + [identity]
        activation_ders  = [derivate(f) for f in activation_funcs]

        opt_bucket = {}  # optimizer -> grid

        # === Iterate over optimizers (INSIDE the activation loop) ===
        for opt_name, (optimizer_class, fixed_params) in optimizers_to_sweep.items():
            print(f"    --- Optimizer {opt_name} ---")

            # Store results for this optimizer across all eta and lambda values
            grid = np.zeros((len(eta_vals), len(lam_vals)))

            for i, eta in enumerate(eta_vals):
                for j, lam in enumerate(lam_vals):
                    # Prepare optimizer kwargs
                    current_kwargs = {'eta': float(eta)}
                    current_kwargs.update(fixed_params)

                    # Fresh network per run
                    nn = FFNN(
                        network_input_size=input_nodes,
                        layer_output_sizes=tuple(layer_sizes),
                        activation_funcs=activation_funcs,
                        activation_ders=activation_ders,
                        cost_fun=CostOLS,
                        cost_der=dCostOLS,
                        seed=42
                    )
                    nn.reset_weights()

                    # Scheduler & batch mode (GD uses full-batch)
                    scheduler_instance = optimizer_class(**current_kwargs)
                    batches_arg = 1 if opt_name == 'GD' else batches

                    # Train with L2 only (L1=0)
                    _scores = nn.fit(
                        X_train_scaled, y_train,
                        scheduler=scheduler_instance,
                        epochs=epochs_sweep,
                        batches=batches_arg,
                        l1=0.0,
                        l2=float(lam)
                    )

                    # Evaluate on fixed test set
                    y_pred_test = nn.predict(X_fixed_test_scaled)
                    final_mse   = CostOLS(y_pred_test.ravel(), y_fixed_test.ravel())

                    grid[i, j] = final_mse
                    print(f"      act={act_name} | opt={opt_name} | eta={eta:.1e}, lam={lam:.1e} -> MSE={final_mse:.6f}")

            opt_bucket[opt_name] = grid

        act_bucket[act_name] = opt_bucket

    L2_sweep_results[arch_name] = act_bucket


# flatten L2_sweep_results 
rows = []
for arch_name, act_dict in L2_sweep_results.items():
    for act_name, opt_dict in act_dict.items():
        for opt_name, grid in opt_dict.items():
            for i, eta in enumerate(eta_vals):
                for j, lam in enumerate(lam_vals):
                    rows.append({
                        "architecture": arch_name,
                        "activation": act_name,
                        "optimizer": opt_name,
                        "eta": float(eta),
                        "lam": float(lam),
                        "mse": float(grid[i, j]),
                    })
l2_df = pd.DataFrame(rows)
l2_df.to_csv("l2_results.csv")  
print("Saved: l2_results.csv")




sizes: 6 4 3 4 4
total combos: 1152
sample size: 120
Total runs: 1152


================ L2 (Ridge) Regularization with Activation Sweep ================

---- Architecture: 0_Hidden_Layers ----
  >> Activation: Sigmoid
    --- Optimizer GD ---
      act=Sigmoid | opt=GD | eta=1.0e-04, lam=1.0e-04 -> MSE=0.169448
      act=Sigmoid | opt=GD | eta=1.0e-04, lam=1.0e-03 -> MSE=0.169448
      act=Sigmoid | opt=GD | eta=1.0e-04, lam=1.0e-02 -> MSE=0.169448
      act=Sigmoid | opt=GD | eta=1.0e-04, lam=0.0e+00 -> MSE=0.169448
      act=Sigmoid | opt=GD | eta=1.0e-03, lam=1.0e-04 -> MSE=0.167001
      act=Sigmoid | opt=GD | eta=1.0e-03, lam=1.0e-03 -> MSE=0.167001
      act=Sigmoid | opt=GD | eta=1.0e-03, lam=1.0e-02 -> MSE=0.167001
      act=Sigmoid | opt=GD | eta=1.0e-03, lam=0.0e+00 -> MSE=0.167001
      act=Sigmoid | opt=GD | eta=1.0e-02, lam=1.0e-04 -> MSE=0.146486
      act=Sigmoid | opt=GD | eta=1.0e-02, lam=1.0e-03 -> MSE=0.146486
      act=Sigmoid | opt=GD | eta=1.0e-02, lam=1.0e-02 -

## L1 regularization

In [43]:
eta_constant = 1e-2 # if fixed

try:
    eta_values = list(eta_constant)   # iterate over several values
except TypeError:
    eta_values = [eta_constant] 


total = len(architectures_to_sweep) * len(optimizers_to_sweep) * len(activation_tests) *len(eta_vals) * len(l1_vals)
print("Total runs:", total)

L1_sweep_results = {}


print("\n\n================  L1 (Lasso) Regularization  ================")
for arch_name, layer_sizes in architectures_to_sweep.items():
    print(f"\n---- Architecture: {arch_name} ----")
    act_bucket = {}  # activation -> optimizer -> (len(eta_values) x len(l1_vals)) grid
    n_hidden_layers = max(0, len(layer_sizes) - 1)

    # === Iterate over activation functions ===
    for act_name, h_func in activation_tests.items():
        print(f"  >> Activation: {act_name}")
        activation_funcs = [h_func]*n_hidden_layers + [identity]
        activation_ders  = [derivate(f) for f in activation_funcs]

        opt_bucket = {}  # optimizer -> grid

        # === Iterate over optimizers ===
        for opt_name, (optimizer_class, fixed_params) in optimizers_to_sweep.items():
            print(f"    --- Optimizer {opt_name} ---")

            # Grid over (eta, L1)
            grid = np.zeros((len(eta_values), len(l1_vals)))
            for i, eta in enumerate(eta_values):
                for j, l1 in enumerate(l1_vals):
                    current_kwargs = {'eta': float(eta)}
                    current_kwargs.update(fixed_params)

                    # Fresh network per run
                    nn = FFNN(
                        network_input_size=input_nodes,
                        layer_output_sizes=tuple(layer_sizes),
                        activation_funcs=activation_funcs,
                        activation_ders=activation_ders,
                        cost_fun=CostOLS, cost_der=dCostOLS,
                        seed=42
                    )
                    nn.reset_weights()

                    scheduler_instance = optimizer_class(**current_kwargs)
                    batches_arg = 1 if opt_name == 'GD' else batches

                    # Train with L1 only (L2=0)
                    _scores = nn.fit(
                        X_train_scaled, y_train,
                        scheduler=scheduler_instance,
                        epochs=epochs_sweep, batches=batches_arg,
                        l1=float(l1), l2=0.0
                    )

                    y_pred_test = nn.predict(X_fixed_test_scaled)
                    final_mse = CostOLS(y_pred_test.ravel(), y_fixed_test.ravel())
                    grid[i, j] = final_mse

                    print(f"      act={act_name} | opt={opt_name} | eta={eta:.1e}, L1={l1:.1e} -> MSE={final_mse:.6f}")

            opt_bucket[opt_name] = grid

        act_bucket[act_name] = opt_bucket

    L1_sweep_results[arch_name] = act_bucket

# ===== Flatten L1_sweep_results to a DataFrame and save =====
rows = []
for arch_name, act_dict in L1_sweep_results.items():
    for act_name, opt_dict in act_dict.items():
        for opt_name, grid in opt_dict.items():
            for i, eta in enumerate(eta_values):
                for j, l1 in enumerate(l1_vals):
                    rows.append({
                        "architecture": arch_name,
                        "activation": act_name,
                        "optimizer": opt_name,
                        "eta": float(eta),
                        "l1": float(l1),
                        "mse": float(grid[i, j]),
                    })
l1_df = pd.DataFrame(rows)
l1_df.to_csv("l1_results.csv", index=False)
print("Saved: l1_results.csv")

Total runs: 1152


================  L1 (Lasso) Regularization  ================

---- Architecture: 0_Hidden_Layers ----
  >> Activation: Sigmoid
    --- Optimizer GD ---
      act=Sigmoid | opt=GD | eta=1.0e-02, L1=1.0e-04 -> MSE=0.146486
      act=Sigmoid | opt=GD | eta=1.0e-02, L1=1.0e-03 -> MSE=0.146485
      act=Sigmoid | opt=GD | eta=1.0e-02, L1=1.0e-02 -> MSE=0.146480
      act=Sigmoid | opt=GD | eta=1.0e-02, L1=0.0e+00 -> MSE=0.146486
    --- Optimizer SGD ---
      act=Sigmoid | opt=SGD | eta=1.0e-02, L1=1.0e-04 -> MSE=0.094824
      act=Sigmoid | opt=SGD | eta=1.0e-02, L1=1.0e-03 -> MSE=0.094804
      act=Sigmoid | opt=SGD | eta=1.0e-02, L1=1.0e-02 -> MSE=0.094634
      act=Sigmoid | opt=SGD | eta=1.0e-02, L1=0.0e+00 -> MSE=0.094826
    --- Optimizer RMS_prop ---
      act=Sigmoid | opt=RMS_prop | eta=1.0e-02, L1=1.0e-04 -> MSE=0.094251
      act=Sigmoid | opt=RMS_prop | eta=1.0e-02, L1=1.0e-03 -> MSE=0.094239
      act=Sigmoid | opt=RMS_prop | eta=1.0e-02, L1=1.0e-02 -> MSE

In [44]:
K = 10  # top 10 runs

# L1
topk_l1 = l1_df.nsmallest(K, "mse").reset_index(drop=True)
best_l1 = l1_df.nsmallest(1, "mse").reset_index(drop=True)

print("\n=== Top-K L1 runs ===")
display(topk_l1)
print("\n=== Absolute best L1 ===")
display(best_l1)

# L2
topk_l2 = l2_df.nsmallest(K, "mse").reset_index(drop=True)
best_l2 = l2_df.nsmallest(1, "mse").reset_index(drop=True)

print("\n=== Top-K L2 runs ===")
display(topk_l2)
print("\n=== Absolute best L2 ===")
display(best_l2)



=== Top-K L1 runs ===


,architecture,activation,optimizer,eta,l1,mse
0,"2_Hidden_Layers (50, 100)",RELU,RMS_prop,0.01,0.0001,0.011134
1,"2_Hidden_Layers (50, 100)",LRELU,RMS_prop,0.01,0.0000,0.011257
2,"2_Hidden_Layers (100, 200)",RELU,RMS_prop,0.01,0.0001,0.011365
3,"2_Hidden_Layers (50, 100)",LRELU,Adam,0.01,0.0000,0.012285
4,"2_Hidden_Layers (50, 100)",RELU,Adam,0.01,0.0000,0.012298
5,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.01,0.0000,0.012322
6,"3_Hidden_Layers (50, 100, 200)",RELU,Adam,0.01,0.0001,0.012428
7,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.01,0.0001,0.012554
8,"2_Hidden_Layers (50, 100)",LRELU,RMS_prop,0.01,0.0001,0.012626
9,"2_Hidden_Layers (50, 100)",RELU,RMS_prop,0.01,0.0000,0.012688



=== Absolute best L1 ===


,architecture,activation,optimizer,eta,l1,mse
0,"2_Hidden_Layers (50, 100)",RELU,RMS_prop,0.01,0.0001,0.011134



=== Top-K L2 runs ===


,architecture,activation,optimizer,eta,lam,mse
0,"3_Hidden_Layers (50, 100, 200)",RELU,Adam,0.010,0.0010,0.010620
1,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.001,0.0001,0.010724
2,"3_Hidden_Layers (50, 100, 200)",RELU,Adam,0.001,0.0000,0.010835
3,"3_Hidden_Layers (50, 100, 200)",RELU,Adam,0.001,0.0001,0.011008
4,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.001,0.0000,0.011042
5,"2_Hidden_Layers (50, 100)",LRELU,RMS_prop,0.010,0.0000,0.011257
6,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.010,0.0010,0.011264
7,"2_Hidden_Layers (50, 100)",LRELU,Adam,0.100,0.0001,0.011802
8,"3_Hidden_Layers (50, 100, 200)",LRELU,RMS_prop,0.001,0.0000,0.011844
9,"3_Hidden_Layers (50, 100, 200)",RELU,RMS_prop,0.001,0.0001,0.011939



=== Absolute best L2 ===


,architecture,activation,optimizer,eta,lam,mse
0,"3_Hidden_Layers (50, 100, 200)",RELU,Adam,0.01,0.001,0.01062


## Retraining and finding the best combo overall

In [45]:
# Make sure this is set (used inside _train_once)
RETRAIN_EPOCHS = 500  # or e.g. epochs_sweep*2

def retrain_table(df_in, reg_type, reg_value_col, repeats=1):
    # Sanity checks
    needed = {"architecture","activation","optimizer","eta","mse",reg_value_col}
    missing = needed - set(df_in.columns)
    if missing:
        raise ValueError(f"Missing columns in df_in: {missing}")

    rows = []
    for i, r in df_in.iterrows():
        arch = r["architecture"]
        act  = r["activation"]
        opt  = r["optimizer"]
        eta  = float(r["eta"])
        old  = float(r["mse"])
        regv = float(r[reg_value_col])

        if repeats == 1:
            new_mse = 
            (arch, act, opt, eta, reg_type, regv)
        else:
            vals = [_train_once(arch, act, opt, eta, reg_type, regv) for _ in range(repeats)]
            new_mse = float(np.mean(vals))

        delta = new_mse - old

        print(f"{i+1:02d}. {arch} | {act} | {opt} | η={eta:.1e} | {reg_type}={regv:.1e} "
              f"-> old={old:.6f} | new={new_mse:.6f} | Δ={delta:+.6f}")

        rows.append({
            "architecture": arch, "activation": act, "optimizer": opt,
            "eta": eta, "reg_type": reg_type, "reg_value": regv,
            "old_mse": old, "new_mse": new_mse, "delta": delta,
            "epochs": RETRAIN_EPOCHS, "repeats": repeats,
        })

    out = pd.DataFrame(rows).sort_values("new_mse").reset_index(drop=True)
    print(f"\n=== Retrain summary ({reg_type}) — top {len(df_in)} ===")
    display(out.head(min(10, len(out))))
    return out

# Retrain Top-K L1 and L2 (where topk_l1 has column 'l1' and topk_l2 has 'lam')
retrained_topk_l1 = retrain_table(topk_l1, reg_type="L1", reg_value_col="l1", repeats=1)
retrained_topk_l2 = retrain_table(topk_l2, reg_type="L2", reg_value_col="lam", repeats=1)


01. 2_Hidden_Layers (50, 100) | RELU | RMS_prop | η=1.0e-02 | L1=1.0e-04 -> old=0.011134 | new=0.011168 | Δ=+0.000033
02. 2_Hidden_Layers (50, 100) | LRELU | RMS_prop | η=1.0e-02 | L1=0.0e+00 -> old=0.011257 | new=0.011423 | Δ=+0.000166
03. 2_Hidden_Layers (100, 200) | RELU | RMS_prop | η=1.0e-02 | L1=1.0e-04 -> old=0.011365 | new=0.012000 | Δ=+0.000634
04. 2_Hidden_Layers (50, 100) | LRELU | Adam | η=1.0e-02 | L1=0.0e+00 -> old=0.012285 | new=0.011035 | Δ=-0.001250
05. 2_Hidden_Layers (50, 100) | RELU | Adam | η=1.0e-02 | L1=0.0e+00 -> old=0.012298 | new=0.011208 | Δ=-0.001090
06. 3_Hidden_Layers (50, 100, 200) | LRELU | Adam | η=1.0e-02 | L1=0.0e+00 -> old=0.012322 | new=0.010961 | Δ=-0.001361
07. 3_Hidden_Layers (50, 100, 200) | RELU | Adam | η=1.0e-02 | L1=1.0e-04 -> old=0.012428 | new=0.010869 | Δ=-0.001559
08. 3_Hidden_Layers (50, 100, 200) | LRELU | Adam | η=1.0e-02 | L1=1.0e-04 -> old=0.012554 | new=0.010805 | Δ=-0.001749
09. 2_Hidden_Layers (50, 100) | LRELU | RMS_prop | η=1.0

,architecture,activation,optimizer,eta,reg_type,reg_value,old_mse,new_mse,delta,epochs,repeats
0,"2_Hidden_Layers (50, 100)",LRELU,RMS_prop,0.01,L1,0.0001,0.012626,0.010511,-0.002114,500,1
1,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.01,L1,0.0001,0.012554,0.010805,-0.001749,500,1
2,"3_Hidden_Layers (50, 100, 200)",RELU,Adam,0.01,L1,0.0001,0.012428,0.010869,-0.001559,500,1
3,"2_Hidden_Layers (50, 100)",RELU,RMS_prop,0.01,L1,0.0000,0.012688,0.010929,-0.001759,500,1
4,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.01,L1,0.0000,0.012322,0.010961,-0.001361,500,1
5,"2_Hidden_Layers (50, 100)",LRELU,Adam,0.01,L1,0.0000,0.012285,0.011035,-0.001250,500,1
6,"2_Hidden_Layers (50, 100)",RELU,RMS_prop,0.01,L1,0.0001,0.011134,0.011168,0.000033,500,1
7,"2_Hidden_Layers (50, 100)",RELU,Adam,0.01,L1,0.0000,0.012298,0.011208,-0.001090,500,1
8,"2_Hidden_Layers (50, 100)",LRELU,RMS_prop,0.01,L1,0.0000,0.011257,0.011423,0.000166,500,1
9,"2_Hidden_Layers (100, 200)",RELU,RMS_prop,0.01,L1,0.0001,0.011365,0.012000,0.000634,500,1


01. 3_Hidden_Layers (50, 100, 200) | RELU | Adam | η=1.0e-02 | L2=1.0e-03 -> old=0.010620 | new=0.010935 | Δ=+0.000315
02. 3_Hidden_Layers (50, 100, 200) | LRELU | Adam | η=1.0e-03 | L2=1.0e-04 -> old=0.010724 | new=0.011332 | Δ=+0.000608
03. 3_Hidden_Layers (50, 100, 200) | RELU | Adam | η=1.0e-03 | L2=0.0e+00 -> old=0.010835 | new=0.011275 | Δ=+0.000440
04. 3_Hidden_Layers (50, 100, 200) | RELU | Adam | η=1.0e-03 | L2=1.0e-04 -> old=0.011008 | new=0.011286 | Δ=+0.000278
05. 3_Hidden_Layers (50, 100, 200) | LRELU | Adam | η=1.0e-03 | L2=0.0e+00 -> old=0.011042 | new=0.011427 | Δ=+0.000384
06. 2_Hidden_Layers (50, 100) | LRELU | RMS_prop | η=1.0e-02 | L2=0.0e+00 -> old=0.011257 | new=0.011423 | Δ=+0.000166
07. 3_Hidden_Layers (50, 100, 200) | LRELU | Adam | η=1.0e-02 | L2=1.0e-03 -> old=0.011264 | new=0.010780 | Δ=-0.000484
08. 2_Hidden_Layers (50, 100) | LRELU | Adam | η=1.0e-01 | L2=1.0e-04 -> old=0.011802 | new=0.014589 | Δ=+0.002788
09. 3_Hidden_Layers (50, 100, 200) | LRELU | RMS_

,architecture,activation,optimizer,eta,reg_type,reg_value,old_mse,new_mse,delta,epochs,repeats
0,"3_Hidden_Layers (50, 100, 200)",RELU,RMS_prop,0.001,L2,0.0001,0.011939,0.010386,-0.001553,500,1
1,"3_Hidden_Layers (50, 100, 200)",LRELU,RMS_prop,0.001,L2,0.0000,0.011844,0.010691,-0.001152,500,1
2,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.010,L2,0.0010,0.011264,0.010780,-0.000484,500,1
3,"3_Hidden_Layers (50, 100, 200)",RELU,Adam,0.010,L2,0.0010,0.010620,0.010935,0.000315,500,1
4,"3_Hidden_Layers (50, 100, 200)",RELU,Adam,0.001,L2,0.0000,0.010835,0.011275,0.000440,500,1
5,"3_Hidden_Layers (50, 100, 200)",RELU,Adam,0.001,L2,0.0001,0.011008,0.011286,0.000278,500,1
6,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.001,L2,0.0001,0.010724,0.011332,0.000608,500,1
7,"2_Hidden_Layers (50, 100)",LRELU,RMS_prop,0.010,L2,0.0000,0.011257,0.011423,0.000166,500,1
8,"3_Hidden_Layers (50, 100, 200)",LRELU,Adam,0.001,L2,0.0000,0.011042,0.011427,0.000384,500,1
9,"2_Hidden_Layers (50, 100)",LRELU,Adam,0.100,L2,0.0001,0.011802,0.014589,0.002788,500,1


In [46]:
# --- Best after retraining (L1) ---
best_l1_idx = retrained_topk_l1["new_mse"].idxmin()
best_l1 = retrained_topk_l1.loc[best_l1_idx]

print("\n=== Best L1 after retraining ===")
print(f"{best_l1['architecture']} | {best_l1['activation']} | {best_l1['optimizer']}"
      f" | η={best_l1['eta']:.1e} | L1={best_l1['reg_value']:.1e}"
      f" -> old={best_l1['old_mse']:.6f} | new={best_l1['new_mse']:.6f}"
      f" | Δ={best_l1['delta']:+.6f}")

# --- Best after retraining (L2) ---
best_l2_idx = retrained_topk_l2["new_mse"].idxmin()
best_l2 = retrained_topk_l2.loc[best_l2_idx]

print("\n=== Best L2 after retraining ===")
print(f"{best_l2['architecture']} | {best_l2['activation']} | {best_l2['optimizer']}"
      f" | η={best_l2['eta']:.1e} | L2={best_l2['reg_value']:.1e}"
      f" -> old={best_l2['old_mse']:.6f} | new={best_l2['new_mse']:.6f}"
      f" | Δ={best_l2['delta']:+.6f}")



=== Best L1 after retraining ===
2_Hidden_Layers (50, 100) | LRELU | RMS_prop | η=1.0e-02 | L1=1.0e-04 -> old=0.012626 | new=0.010511 | Δ=-0.002114

=== Best L2 after retraining ===
3_Hidden_Layers (50, 100, 200) | RELU | RMS_prop | η=1.0e-03 | L2=1.0e-04 -> old=0.011939 | new=0.010386 | Δ=-0.001553
